# 💊 알약 객체 검출 데이터셋 탐색적 데이터 분석 (EDA)

이 노트북에서는 경구약제 이미지 데이터셋의 구조와 특성을 심층적으로 분석합니다.

**분석 목차:**
1. 데이터 로드 및 구조 파악
2. 이미지 통계 분석
3. 클래스(카테고리) 분포 분석
4. 바운딩 박스 분석
5. 알약 속성 분석 (색상, 모양, 방향 등)
6. 이미지 시각화
7. 데이터 품질 점검
8. 인사이트 요약

In [ ]:
# ─── 라이브러리 임포트 ───────────────────────────────────────────────────────
import os
import json
import glob
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib import font_manager, rc
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['axes.unicode_minus'] = False
try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        font_manager.fontManager.addfont(font_path)
        rc('font', family='NanumGothic')
    else:
        # 대체: DejaVu
        rc('font', family='DejaVu Sans')
except:
    pass

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('Set2', 12)

print('라이브러리 로드 완료 ✓')

In [ ]:
# ─── 경로 설정 ───────────────────────────────────────────────────────────────
BASE_DIR    = Path('data')          # 압축 해제 루트
ANNO_DIR    = BASE_DIR / 'train_annotations'
IMAGE_DIR   = BASE_DIR / 'train_images'

# 압축 해제가 안 되어있으면 해제
import zipfile
if not ANNO_DIR.exists():
    with zipfile.ZipFile('train_annotations.zip', 'r') as z:
        z.extractall(BASE_DIR)
if not IMAGE_DIR.exists():
    with zipfile.ZipFile('train_images.zip', 'r') as z:
        z.extractall(BASE_DIR)

json_files  = sorted(glob.glob(str(ANNO_DIR / '**/*.json'), recursive=True))
image_files = sorted(glob.glob(str(IMAGE_DIR / '**/*.png'), recursive=True))

print(f'JSON 파일 수  : {len(json_files):,}')
print(f'이미지 파일 수: {len(image_files):,}')

## 1. 데이터 로드 및 구조 파악

In [ ]:
# ─── 전체 JSON 파싱 ──────────────────────────────────────────────────────────
all_images      = []
all_annotations = []
all_categories  = {}  # id -> name

for fpath in json_files:
    with open(fpath, 'r', encoding='utf-8') as fp:
        data = json.load(fp)

    for img in data.get('images', []):
        img['_json_path'] = fpath          # 출처 JSON 저장
        all_images.append(img)

    for ann in data.get('annotations', []):
        bbox = ann.get('bbox', [])
        if bbox and len(bbox) == 4:        # 유효한 bbox만
            all_annotations.append(ann)

    for cat in data.get('categories', []):
        all_categories[cat['id']] = cat['name']

# DataFrame 변환
df_images = pd.DataFrame(all_images)
df_annot  = pd.DataFrame(all_annotations)
df_annot[['bbox_x','bbox_y','bbox_w','bbox_h']] = pd.DataFrame(
    df_annot['bbox'].tolist(), index=df_annot.index
)
df_annot['area'] = df_annot['bbox_w'] * df_annot['bbox_h']
df_annot['aspect_ratio'] = df_annot['bbox_w'] / df_annot['bbox_h']

print(f'이미지  DataFrame: {df_images.shape}')
print(f'어노테이션 DataFrame: {df_annot.shape}')
print(f'카테고리 수: {len(all_categories)}')
df_images.head(2)

In [ ]:
# ─── 기본 통계 요약 ──────────────────────────────────────────────────────────
print('=== 이미지 데이터 요약 ===')
print(df_images[['width','height','thick','leng_long','leng_short']].describe().round(2))

print('\n=== 어노테이션 요약 ===')
print(df_annot[['bbox_x','bbox_y','bbox_w','bbox_h','area','aspect_ratio']].describe().round(2))

## 2. 이미지 통계 분석

In [ ]:
# ─── 이미지 해상도 & 객체 수 분포 ───────────────────────────────────────────
ann_per_img = df_annot.groupby('image_id').size().reset_index(name='pill_count')
df_images   = df_images.merge(ann_per_img, left_on='id', right_on='image_id', how='left')
df_images['pill_count'] = df_images['pill_count'].fillna(0).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 이미지 크기
axes[0].scatter(df_images['width'], df_images['height'],
                alpha=0.5, color=PALETTE[0], edgecolors='white', s=50)
axes[0].set_title('이미지 해상도 분포', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Height (px)')

# 이미지당 알약 수
pill_dist = df_images['pill_count'].value_counts().sort_index()
bars = axes[1].bar(pill_dist.index.astype(str), pill_dist.values,
                   color=PALETTE[1], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, pill_dist.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val}\n({val/len(df_images)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11)
axes[1].set_title('이미지당 알약 개수 분포', fontsize=14, fontweight='bold')
axes[1].set_xlabel('알약 수'); axes[1].set_ylabel('이미지 수')

# 카메라 각도(위도) 분포
cam_la_dist = df_images['camera_la'].value_counts().sort_index()
axes[2].bar(cam_la_dist.index.astype(str), cam_la_dist.values,
            color=PALETTE[2], edgecolor='white', linewidth=1.2)
axes[2].set_title('카메라 위도(촬영각도) 분포', fontsize=14, fontweight='bold')
axes[2].set_xlabel('camera_la (도)'); axes[2].set_ylabel('이미지 수')

plt.tight_layout()
plt.savefig('eda_image_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n전체 이미지 해상도: {set(zip(df_images.width, df_images.height))}')
print(f'카메라 각도 분포  : {cam_la_dist.to_dict()}')

## 3. 클래스(카테고리) 분포 분석

In [ ]:
# ─── 카테고리별 어노테이션 수 ─────────────────────────────────────────────────
df_annot['category_name'] = df_annot['category_id'].map(all_categories).fillna('Unknown')
cat_counts = df_annot['category_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 전체 카테고리 바차트
top_n = 30
top_cats = cat_counts.head(top_n)
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.2, top_n))
axes[0].barh(top_cats.index[::-1], top_cats.values[::-1], color=colors[::-1])
axes[0].set_title(f'카테고리별 어노테이션 수 (상위 {top_n}개)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('어노테이션 수')
for i, (val, name) in enumerate(zip(top_cats.values[::-1], top_cats.index[::-1])):
    axes[0].text(val + 0.5, i, str(val), va='center', fontsize=9)

# 클래스 불균형 — 로그 스케일
axes[1].bar(range(len(cat_counts)), cat_counts.values,
            color=PALETTE[3], edgecolor='none', alpha=0.8)
axes[1].set_yscale('log')
axes[1].set_title('카테고리 분포 (로그 스케일)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('카테고리 인덱스'); axes[1].set_ylabel('어노테이션 수 (log)')

plt.tight_layout()
plt.savefig('eda_category_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n총 카테고리 수  : {len(cat_counts)}')
print(f'최다 클래스     : {cat_counts.idxmax()} ({cat_counts.max()}개)')
print(f'최소 클래스     : {cat_counts.idxmin()} ({cat_counts.min()}개)')
print(f'클래스 불균형비 : {cat_counts.max()/cat_counts.min():.1f}x')

In [ ]:
# ─── 클래스별 어노테이션 수 히스토그램 ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cat_counts.values, bins=20, color=PALETTE[4], edgecolor='white', linewidth=1)
ax.axvline(cat_counts.mean(), color='red', linestyle='--', label=f'평균: {cat_counts.mean():.1f}')
ax.axvline(cat_counts.median(), color='blue', linestyle='--', label=f'중앙값: {cat_counts.median():.1f}')
ax.set_title('클래스당 어노테이션 수 분포', fontsize=14, fontweight='bold')
ax.set_xlabel('어노테이션 수'); ax.set_ylabel('클래스 수')
ax.legend()
plt.tight_layout()
plt.savefig('eda_class_hist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 바운딩 박스 분석

In [ ]:
# ─── BBox 크기 및 위치 분석 ──────────────────────────────────────────────────
IMG_W, IMG_H = 976, 1280

# 정규화
df_annot['rel_w']       = df_annot['bbox_w'] / IMG_W
df_annot['rel_h']       = df_annot['bbox_h'] / IMG_H
df_annot['center_x']    = (df_annot['bbox_x'] + df_annot['bbox_w']/2) / IMG_W
df_annot['center_y']    = (df_annot['bbox_y'] + df_annot['bbox_h']/2) / IMG_H
df_annot['rel_area']    = df_annot['rel_w'] * df_annot['rel_h']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# BBox 너비 분포
axes[0,0].hist(df_annot['bbox_w'], bins=30, color=PALETTE[0], edgecolor='white')
axes[0,0].set_title('BBox 너비 분포', fontweight='bold'); axes[0,0].set_xlabel('Width (px)')

# BBox 높이 분포
axes[0,1].hist(df_annot['bbox_h'], bins=30, color=PALETTE[1], edgecolor='white')
axes[0,1].set_title('BBox 높이 분포', fontweight='bold'); axes[0,1].set_xlabel('Height (px)')

# 가로세로 비율
axes[0,2].hist(df_annot['aspect_ratio'], bins=30, color=PALETTE[2], edgecolor='white')
axes[0,2].axvline(1.0, color='red', linestyle='--', label='1:1')
axes[0,2].set_title('종횡비(W/H) 분포', fontweight='bold'); axes[0,2].set_xlabel('Aspect Ratio')
axes[0,2].legend()

# BBox 면적 분포
axes[1,0].hist(df_annot['rel_area'], bins=30, color=PALETTE[3], edgecolor='white')
axes[1,0].set_title('상대 면적 분포 (BBox/Image)', fontweight='bold')
axes[1,0].set_xlabel('Relative Area')

# 너비 vs 높이 산점도
sc = axes[1,1].scatter(df_annot['bbox_w'], df_annot['bbox_h'],
                       c=df_annot['rel_area'], cmap='YlOrRd', alpha=0.6, s=40)
plt.colorbar(sc, ax=axes[1,1], label='상대 면적')
axes[1,1].set_title('BBox 너비 vs 높이', fontweight='bold')
axes[1,1].set_xlabel('Width'); axes[1,1].set_ylabel('Height')

# 중심점 히트맵
h, xedges, yedges = np.histogram2d(
    df_annot['center_x'], df_annot['center_y'], bins=20,
    range=[[0,1],[0,1]]
)
im = axes[1,2].imshow(h.T, origin='lower', cmap='hot', extent=[0,1,0,1], aspect='auto')
plt.colorbar(im, ax=axes[1,2])
axes[1,2].set_title('BBox 중심점 히트맵', fontweight='bold')
axes[1,2].set_xlabel('X (정규화)'); axes[1,2].set_ylabel('Y (정규화)')

plt.tight_layout()
plt.savefig('eda_bbox_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'BBox W: {df_annot.bbox_w.min():.0f} ~ {df_annot.bbox_w.max():.0f} px (평균 {df_annot.bbox_w.mean():.1f})')
print(f'BBox H: {df_annot.bbox_h.min():.0f} ~ {df_annot.bbox_h.max():.0f} px (평균 {df_annot.bbox_h.mean():.1f})')
print(f'종횡비: {df_annot.aspect_ratio.min():.2f} ~ {df_annot.aspect_ratio.max():.2f} (평균 {df_annot.aspect_ratio.mean():.2f})')
print(f'상대 면적: {df_annot.rel_area.min():.3f} ~ {df_annot.rel_area.max():.3f} (평균 {df_annot.rel_area.mean():.3f})')

## 5. 알약 속성 분석 (색상, 모양, 방향 등)

In [ ]:
# ─── 색상·모양·방향 분포 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

def plot_bar(ax, series, title, color_list=None, top_n=None):
    vc = series.value_counts().dropna()
    if top_n: vc = vc.head(top_n)
    colors = color_list if color_list else PALETTE[:len(vc)]
    bars = ax.bar(vc.index, vc.values, color=colors[:len(vc)], edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{v}\n({v/vc.sum()*100:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('이미지 수')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=10)

# 색상 매핑 (실제 색상으로)
color_map = {
    '하양':'#EEEEEE','주황':'#FF8C00','분홍':'#FFB6C1','노랑':'#FFD700',
    '갈색':'#8B4513','초록':'#3CB371','파랑':'#4169E1','보라':'#9370DB',
    '빨강':'#DC143C','회색':'#A9A9A9','검정':'#2F4F4F','연두':'#90EE90'
}

color1_vc = df_images['color_class1'].value_counts().dropna().head(12)
bar_colors = [color_map.get(c, '#AAAAAA') for c in color1_vc.index]
bars = axes[0,0].bar(color1_vc.index, color1_vc.values, color=bar_colors,
                      edgecolor='gray', linewidth=1)
for bar, v in zip(bars, color1_vc.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   str(v), ha='center', va='bottom', fontsize=10)
axes[0,0].set_title('알약 색상(color_class1) 분포', fontsize=14, fontweight='bold')
axes[0,0].set_ylabel('이미지 수')
plt.setp(axes[0,0].get_xticklabels(), rotation=30, ha='right')

# 모양
plot_bar(axes[0,1], df_images['drug_shape'], '알약 모양(drug_shape) 분포',
         color_list=list(PALETTE))

# 제형 분류
plot_bar(axes[1,0], df_images['form_code_name'], '제형 분류(form_code_name) 분포',
         color_list=list(PALETTE))

# 앞면/뒷면 분류
dir_vc = df_images['drug_dir'].value_counts()
axes[1,1].pie(dir_vc.values, labels=dir_vc.index, autopct='%1.1f%%',
              colors=[PALETTE[5], PALETTE[6]], startangle=90,
              textprops={'fontsize': 13})
axes[1,1].set_title('알약 방향(drug_dir) 분포', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_pill_attrs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 알약 물리적 크기 (두께, 장축, 단축) ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label, color in zip(
    axes,
    ['thick', 'leng_long', 'leng_short'],
    ['두께 (mm)', '장축 (mm)', '단축 (mm)'],
    PALETTE[:3]
):
    vals = df_images[col].dropna()
    ax.hist(vals, bins=25, color=color, edgecolor='white', linewidth=1)
    ax.axvline(vals.mean(), color='red', linestyle='--', label=f'평균 {vals.mean():.1f}')
    ax.set_title(f'알약 {label} 분포', fontsize=14, fontweight='bold')
    ax.set_xlabel(label); ax.set_ylabel('빈도')
    ax.legend()

plt.tight_layout()
plt.savefig('eda_pill_size.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 전문/일반 의약품 & 분류 분포 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 전문/일반
etc_vc = df_images['di_etc_otc_code'].value_counts()
axes[0].pie(etc_vc.values, labels=etc_vc.index, autopct='%1.1f%%',
            colors=[PALETTE[0], PALETTE[1]], startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('전문/일반 의약품 비율', fontsize=14, fontweight='bold')

# 약품 분류 (상위 15)
class_vc = df_images['di_class_no'].value_counts().head(15)
axes[1].barh(class_vc.index[::-1], class_vc.values[::-1], color=PALETTE[2])
axes[1].set_title('약품 분류(di_class_no) 상위 15', fontsize=14, fontweight='bold')
axes[1].set_xlabel('이미지 수')
for i, v in enumerate(class_vc.values[::-1]):
    axes[1].text(v+0.2, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_drug_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 이미지 시각화

In [ ]:
# ─── 이미지 + BBox 시각화 헬퍼 ───────────────────────────────────────────────
def draw_bboxes(image_id, ax, title=None):
    img_meta = df_images[df_images['id'] == image_id].iloc[0]
    fname    = img_meta['file_name']

    # 이미지 파일 찾기
    matches  = glob.glob(str(IMAGE_DIR / f'**/{fname}'), recursive=True)
    if not matches:
        ax.text(0.5, 0.5, f'이미지 없음:\n{fname}',
                ha='center', va='center', transform=ax.transAxes)
        return

    img = Image.open(matches[0]).convert('RGB')
    ax.imshow(img)

    anns = df_annot[df_annot['image_id'] == image_id]
    colors_cycle = plt.cm.Set1(np.linspace(0, 1, max(len(anns), 1)))

    for (_, ann), col in zip(anns.iterrows(), colors_cycle):
        x, y, w, h = ann['bbox_x'], ann['bbox_y'], ann['bbox_w'], ann['bbox_h']
        rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                  edgecolor=col, facecolor='none')
        ax.add_patch(rect)
        cat_name = all_categories.get(ann['category_id'], str(ann['category_id']))
        ax.text(x, y-5, cat_name, color=col, fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

    ax.set_title(title or fname, fontsize=10)
    ax.axis('off')

# 샘플 이미지 렌더링
sample_ids = df_images['id'].sample(9, random_state=42).tolist()
fig, axes  = plt.subplots(3, 3, figsize=(18, 20))

for ax, img_id in zip(axes.flatten(), sample_ids):
    draw_bboxes(img_id, ax)

plt.suptitle('샘플 이미지 + 바운딩 박스 시각화', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 알약 수별 예시 시각화 ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, pill_n in zip(axes, [2, 3, 4]):
    subset = df_images[df_images['pill_count'] == pill_n]
    if len(subset) == 0:
        continue
    img_id = subset['id'].sample(1, random_state=7).iloc[0]
    draw_bboxes(img_id, ax, title=f'알약 {pill_n}개 예시')

plt.suptitle('이미지당 알약 수별 예시', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_pill_count_examples.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 데이터 품질 점검

In [ ]:
# ─── 결측값 및 이상값 점검 ───────────────────────────────────────────────────
print('=== 이미지 메타데이터 결측값 비율 (%) ===')
miss_ratio = df_images.isnull().mean() * 100
miss_ratio = miss_ratio[miss_ratio > 0].sort_values(ascending=False)
print(miss_ratio.round(2))

print('\n=== BBox 이상값 점검 ===')
# x + w > IMG_W 혹은 y + h > IMG_H 인 케이스
out_of_bounds = df_annot[
    (df_annot['bbox_x'] + df_annot['bbox_w'] > IMG_W) |
    (df_annot['bbox_y'] + df_annot['bbox_h'] > IMG_H)
]
print(f'  이미지 경계 초과 BBox: {len(out_of_bounds)}개')

# 면적 0 이하
zero_area = df_annot[(df_annot['bbox_w'] <= 0) | (df_annot['bbox_h'] <= 0)]
print(f'  면적 0 이하 BBox     : {len(zero_area)}개')

# 중복 image_id
dup_imgs = df_images[df_images.duplicated('id')]
print(f'  중복 image_id        : {len(dup_imgs)}개')

print('\n=== 이미지 파일 존재 여부 점검 ===')
existing  = {Path(p).name for p in image_files}
missing   = df_images[~df_images['file_name'].isin(existing)]
print(f'  총 이미지 메타       : {len(df_images)}')
print(f'  실제 파일 있음       : {len(df_images) - len(missing)}')
print(f'  파일 없음(누락)      : {len(missing)}')

In [ ]:
# ─── BBox 크기 분위수 기반 이상값 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes, ['bbox_w','bbox_h'], ['너비','높이']):
    q1, q3 = df_annot[col].quantile([0.25, 0.75])
    iqr     = q3 - q1
    lower   = q1 - 1.5 * iqr
    upper   = q3 + 1.5 * iqr
    outliers = df_annot[(df_annot[col] < lower) | (df_annot[col] > upper)]
    ax.boxplot(df_annot[col], vert=False, patch_artist=True,
               boxprops=dict(facecolor=PALETTE[0], alpha=0.7))
    ax.set_title(f'BBox {label} 박스플롯 (이상값: {len(outliers)}개)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(f'{label} (px)')

plt.tight_layout()
plt.savefig('eda_bbox_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 인사이트 요약

In [ ]:
# ─── 상관관계 히트맵 ─────────────────────────────────────────────────────────
numeric_cols = ['bbox_w','bbox_h','area','aspect_ratio','rel_w','rel_h','rel_area','center_x','center_y']
corr = df_annot[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('BBox 속성 상관관계 히트맵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 최종 인사이트 요약 ─────────────────────────────────────────────────────
print('=' * 60)
print('           📊 EDA 핵심 인사이트 요약')
print('=' * 60)

print(f"""
📌 데이터셋 규모
   • 전체 이미지    : {len(df_images):,}장
   • 전체 어노테이션: {len(df_annot):,}개
   • 카테고리 수    : {len(all_categories)}개 (테스트셋 40개)

📌 이미지 특성
   • 해상도         : 일관 976×1280 px
   • 배경           : 모두 연회색
   • 조명           : 모두 주백색
   • 촬영방향       : 앞면 {df_images[df_images.drug_dir=='앞면'].shape[0]}장 / 뒷면 {df_images[df_images.drug_dir=='뒷면'].shape[0]}장

📌 알약 분포
   • 이미지당 알약 수: 2~4개
   • 최다 구성       : 3개({df_images[df_images.pill_count==3].shape[0]}장)
   • 최소 구성       : 2개({df_images[df_images.pill_count==2].shape[0]}장)

📌 바운딩 박스
   • 크기 범위       : W {df_annot.bbox_w.min():.0f}~{df_annot.bbox_w.max():.0f}px / H {df_annot.bbox_h.min():.0f}~{df_annot.bbox_h.max():.0f}px
   • 평균 크기       : {df_annot.bbox_w.mean():.0f}×{df_annot.bbox_h.mean():.0f}px
   • 종횡비          : 거의 1:1 (원형·타원 위주)

📌 알약 속성
   • 주요 색상       : 하양·주황·분홍·노랑
   • 주요 모양       : 원형({df_images[df_images.drug_shape=='원형'].shape[0]}) / 타원형({df_images[df_images.drug_shape=='타원형'].shape[0]}) / 장방형({df_images[df_images.drug_shape=='장방형'].shape[0]})
   • 클래스 불균형   : 최다 {cat_counts.max()}개 vs 최소 {cat_counts.min()}개 → {cat_counts.max()/cat_counts.min():.1f}배

📌 모델링 시 주의 사항
   ① 클래스 불균형 → Focal Loss / Class Weighting 고려
   ② 한 이미지에 최대 4개 객체 → Multi-object detection 필수
   ③ 배경·조명 고정 → 색상 지터링 증강으로 일반화 향상 필요
   ④ 고해상도(976×1280) → 적절한 입력 크기 조정 및 Tiling 고려
   ⑤ Train에만 존재하는 클래스 있음 → Test 40개 클래스 집중
""")